# Demo 05 : Events and synchronisation

Run the cells in order and watch. There is nothing to write.

Run the next cell first. It loads the course key and builds `llm`, the course model. Nothing below it depends on where the notebook runs.

In [1]:
# Course setup : run this cell first.
# On the classroom laptop the key comes from .env in the repository root.
# In Colab, add it under the key icon in the left sidebar as COURSE_API_KEY.
import os
import sys
import subprocess
import warnings
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
COURSE_FILES = "https://raw.githubusercontent.com/FeikoWielsma/Multi-Agents-LangGraph/main/"
ROOT = None

if IN_COLAB:
    # Pinned to match uv.lock ; generated by build/stamp_setup.py
    PINS = ["langchain[mcp]==1.4.5", "langchain-core==1.6.10", "langchain-openai==1.7.1", "langgraph==1.2.15", "langgraph-checkpoint-sqlite==3.1.2", "deepagents==0.7.24", "langsmith==0.14.8", "openai==3.29.0", "python-dotenv==1.2.4", "langchain-huggingface==1.2.3", "sentence-transformers==6.1.0", "langchain-text-splitters==1.1.3", "rank-bm25==0.2.2", "agentevals==0.0.9", "httpx==0.28.1", "pytest==9.1.1", "pandas==3.0.6"]
    if not os.getenv("COURSE_SKIP_INSTALL"):
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", *PINS], check=True)
    try:
        from google.colab import userdata
        os.environ.setdefault("COURSE_API_KEY", userdata.get("COURSE_API_KEY"))
    except Exception:
        pass
else:
    from dotenv import load_dotenv
    for folder in [Path.cwd(), *Path.cwd().parents]:
        if (folder / "pyproject.toml").exists():
            ROOT = folder
            load_dotenv(folder / ".env")
            break

PROVIDER = os.getenv("COURSE_PROVIDER", "course")      # "course" or "ollama"
if PROVIDER == "ollama":
    MODEL = os.getenv("COURSE_MODEL", "qwen3.5:4b")
    BASE_URL = "http://localhost:11434/v1"
    API_KEY = "ollama"
    THINKING_OFF = {"reasoning_effort": "none"}
else:
    MODEL = os.getenv("COURSE_MODEL", "openai/gpt-6-luna")  # xiaomi/mimo-v2.6-pro is the step up
    BASE_URL = os.getenv("COURSE_BASE_URL", "https://openrouter.ai/api/v1")
    API_KEY = os.getenv("COURSE_API_KEY")
    THINKING_OFF = {"reasoning": {"effort": "none"}}
    if not API_KEY:
        raise RuntimeError("COURSE_API_KEY is missing : copy .env.example to .env and paste the key")

from langchain_core._api import LangChainBetaWarning
warnings.filterwarnings("ignore", category=LangChainBetaWarning)

from langchain_openai import ChatOpenAI
from langchain_core.rate_limiters import InMemoryRateLimiter

# New OpenRouter accounts may send 20 requests a minute per model.
# One shared limiter keeps parallel agents under it ; raise COURSE_RPS when the cap lifts
RATE_LIMITER = InMemoryRateLimiter(requests_per_second=float(os.getenv("COURSE_RPS", "0.3")),
                                   check_every_n_seconds=0.1, max_bucket_size=1)


def make_llm(**kw):
    """The course chat model. Keyword arguments pass straight to ChatOpenAI."""
    kw.setdefault("model", MODEL)
    kw.setdefault("temperature", 0)
    kw.setdefault("extra_body", THINKING_OFF)
    kw.setdefault("timeout", 90)        # seconds per request ; a stuck call raises instead of hanging
    kw.setdefault("rate_limiter", RATE_LIMITER)
    return ChatOpenAI(base_url=BASE_URL, api_key=API_KEY, **kw)


def make_embeddings(**kw):
    """Local sentence embeddings, so retrieval costs no API calls."""
    from langchain_huggingface import HuggingFaceEmbeddings
    kw.setdefault("model_name", "sentence-transformers/all-MiniLM-L6-v2")
    return HuggingFaceEmbeddings(**kw)


def data(name):
    """Path to a course data file ; downloaded when there is no local copy."""
    if ROOT and (ROOT / "data" / name).exists():
        return str(ROOT / "data" / name)
    target = Path(name)
    if not target.exists():
        import urllib.request
        target.parent.mkdir(parents=True, exist_ok=True)
        urllib.request.urlretrieve(COURSE_FILES + "data/" + name, target)
    return str(target)


def data_folder(name):
    """Path to a course data folder ; downloaded file by file, using its index.txt, when there is no local copy."""
    if ROOT and (ROOT / "data" / name).is_dir():
        return str(ROOT / "data" / name)
    for file in Path(data(f"{name}/index.txt")).read_text(encoding="utf-8").split():
        data(f"{name}/{file}")
    return name


llm = make_llm()
print(f"model {MODEL} via {PROVIDER}")

model openai/gpt-6-luna via course


In [2]:
import json
import operator
import sqlite3
import threading
import time
from pathlib import Path
from typing import Annotated
from typing_extensions import TypedDict

from langchain.messages import AIMessage, HumanMessage, SystemMessage, ToolMessage
from langchain_core.callbacks import BaseCallbackHandler
from langgraph.graph import END, START, MessagesState, StateGraph
from langgraph.types import Command


class Counter(BaseCallbackHandler):
    def __init__(self):
        self.lock = threading.Lock()
        self.calls = self.input_tokens = self.output_tokens = 0

    def on_chat_model_start(self, serialized, messages, **kwargs):
        with self.lock:
            self.calls += 1

    def on_llm_end(self, response, **kwargs):
        with self.lock:
            for batch in response.generations:
                for generation in batch:
                    usage = generation.message.usage_metadata or {}
                    self.input_tokens += usage.get("input_tokens", 0)
                    self.output_tokens += usage.get("output_tokens", 0)


counter = Counter()
llm = make_llm(callbacks=[counter], max_tokens=500, max_retries=0)
started = time.perf_counter()
DB = Path(data("demo-world/service.db")).resolve()
MANUALS = Path(data_folder("demo-world/manuals"))
NOTEBOOK_DIR = ROOT / "notebooks" / "06-agent-communication" if ROOT else Path.cwd()


def stock(part="CE-CF-02"):
    with sqlite3.connect(f"file:{DB.as_posix()}?mode=ro", uri=True) as db:
        db.row_factory = sqlite3.Row
        row = db.execute("SELECT part_no, name, stock, price_eur, depot FROM parts WHERE part_no = ?", (part,)).fetchone()
    return dict(row) if row else {"part_no": part, "error": "unknown part"}


def guidance(error="F-CE-110"):
    return "\n".join(f"[{p.name}] {line}" for p in sorted(MANUALS.glob("*.md"))
                     for line in p.read_text(encoding="utf-8").splitlines()
                     if line.startswith("| " + error + " |"))


FACTS = stock()
QUESTION = "Report CE-CF-02 stock and location, then the first action and replacement condition for F-CE-110."
LIMIT = {"recursion_limit": 16}


def finish_metrics(name):
    row = {"model": MODEL, "rps": float(os.environ.get("COURSE_RPS", "0.3")),
           "calls": counter.calls, "input_tokens": counter.input_tokens,
           "output_tokens": counter.output_tokens, "seconds": round(time.perf_counter() - started, 1)}
    print("metrics :", row)
    (NOTEBOOK_DIR / (name + "-metrics.json")).write_text(json.dumps(row, indent=2) + "\n", encoding="utf-8")

## a · streaming-subgraphs

We stream a compiled parts subgraph and its parent. Namespaces identify the child, while updates contain only the values returned by each node.

In [3]:
class StreamState(TypedDict):
    part_no: str
    finding: dict
    answer: str


child = StateGraph(StreamState)
child.add_node("lookup", lambda s: {"finding": stock(s["part_no"])})
child.add_edge(START, "lookup")
child.add_edge("lookup", END)
parent = StateGraph(StreamState)
parent.add_node("parts", child.compile())
parent.add_node("report", lambda s: {"answer": llm.invoke("Write one stock sentence.\n" + json.dumps(s["finding"])).content})
parent.add_edge(START, "parts")
parent.add_edge("parts", "report")
parent.add_edge("report", END)
graph = parent.compile()
for part in graph.stream({"part_no": "CE-CF-02"}, stream_mode="updates", subgraphs=True, version="v2"):
    print("event :", part["type"], part["ns"], part["data"])

event : updates ('parts:eceb28fd-33ed-9c1c-83cb-d2d15085902d',) {'lookup': {'finding': {'part_no': 'CE-CF-02', 'name': 'traction cooling fan', 'stock': 6, 'price_eur': 420.0, 'depot': 'Rietpoort'}}}
event : updates () {'parts': {'part_no': 'CE-CF-02', 'finding': {'part_no': 'CE-CF-02', 'name': 'traction cooling fan', 'stock': 6, 'price_eur': 420.0, 'depot': 'Rietpoort'}}}


event : updates () {'report': {'answer': 'Rietpoort has 6 traction cooling fans (part no. CE-CF-02) in stock at €420.00 each.'}}


We inspect experimental v3 subgraph projections on a deterministic parent to avoid repeating the model call. We consume each child handle inside the discovery loop before the parent advances.

In [4]:
projection_parent = StateGraph(StreamState)
projection_parent.add_node("parts", child.compile())
projection_parent.add_edge(START, "parts")
projection_parent.add_edge("parts", END)
run = projection_parent.compile().stream_events({"part_no": "CE-CF-02"}, version="v3")
for subrun in run.subgraphs:
    print("v3 path :", subrun.path)
    for value in subrun.values:
        print("v3 child value :", value)
    print("v3 child status :", subrun.status)
print("v3 output :", run.output)

v3 path : ('parts:e5965add-2c80-7d21-372a-6cc1244ac3d3',)
v3 child value : {'part_no': 'CE-CF-02', 'finding': {'part_no': 'CE-CF-02', 'name': 'traction cooling fan', 'stock': 6, 'price_eur': 420.0, 'depot': 'Rietpoort'}}
v3 child status : completed
v3 output : {'part_no': 'CE-CF-02', 'finding': {'part_no': 'CE-CF-02', 'name': 'traction cooling fan', 'stock': 6, 'price_eur': 420.0, 'depot': 'Rietpoort'}}


## b · custom-events

We publish lookup progress through get_stream_writer. A custom event is an observation for a consumer. Only the returned update changes graph state.

In [5]:
from langgraph.config import get_stream_writer


def lookup_with_events(state):
    writer = get_stream_writer()
    writer({"event": "lookup-started", "part_no": state["part_no"]})
    finding = stock(state["part_no"])
    writer({"event": "lookup-finished", "part_no": state["part_no"], "found": "error" not in finding})
    return {"finding": finding}


events = StateGraph(StreamState)
events.add_node("lookup", lookup_with_events)
events.add_edge(START, "lookup")
events.add_edge("lookup", END)
for part in events.compile().stream({"part_no": "CE-CF-02"}, stream_mode=["custom", "updates"], version="v2"):
    print("event :", part["type"], part["data"])

event : custom {'event': 'lookup-started', 'part_no': 'CE-CF-02'}
event : custom {'event': 'lookup-finished', 'part_no': 'CE-CF-02', 'found': True}
event : updates {'lookup': {'finding': {'part_no': 'CE-CF-02', 'name': 'traction cooling fan', 'stock': 6, 'price_eur': 420.0, 'depot': 'Rietpoort'}}}


## c · broadcast-channel

We publish an inventory snapshot once. Each parallel role reads the same broadcast field and returns its own observation through a reducer.

In [6]:
class Broadcast(TypedDict):
    broadcast: dict
    observations: Annotated[list[dict], operator.add]


def subscriber(role):
    def node(state):
        return {"observations": [{"role": role, "seen": state["broadcast"]}]}
    return node


broadcast = StateGraph(Broadcast)
broadcast.add_node("publish", lambda s: {"broadcast": {"version": "BW-7001-stock", "facts": stock()}})
for role in ["seller", "buyer"]:
    broadcast.add_node(role, subscriber(role))
    broadcast.add_edge("publish", role)
    broadcast.add_edge(role, END)
broadcast.add_edge(START, "publish")
result = broadcast.compile().invoke({"observations": []})
for observation in result["observations"]:
    print("broadcast read :", observation)
print("same snapshot :", result["observations"][0]["seen"] == result["observations"][1]["seen"])

broadcast read : {'role': 'buyer', 'seen': {'version': 'BW-7001-stock', 'facts': {'part_no': 'CE-CF-02', 'name': 'traction cooling fan', 'stock': 6, 'price_eur': 420.0, 'depot': 'Rietpoort'}}}
broadcast read : {'role': 'seller', 'seen': {'version': 'BW-7001-stock', 'facts': {'part_no': 'CE-CF-02', 'name': 'traction cooling fan', 'stock': 6, 'price_eur': 420.0, 'depot': 'Rietpoort'}}}
same snapshot : True


The channel lives in one graph run. A branch reads the snapshot published before it starts, and it does not see another branch's in-progress writes. A real stock action needs a current authorised check.

## d · fan-in-and-barriers

We compare an explicit join for fixed dependencies with a deferred collector for branches of unequal depth. The collector checks the required results before writing a final note.

In [7]:
class Joined(TypedDict):
    findings: Annotated[list[dict], operator.add]
    collected: list[str]


def collect(state):
    roles = sorted(item["role"] for item in state["findings"])
    if roles != ["manuals", "parts"]:
        raise ValueError("Required branches are incomplete.")
    return {"collected": roles}


def join_graph(deferred=False):
    b = StateGraph(Joined)
    b.add_node("parts", lambda s: {"findings": [{"role": "parts", "data": stock()}]})
    b.add_node("prepare_manuals", lambda s: {})
    b.add_node("manuals", lambda s: {"findings": [{"role": "manuals", "data": guidance()}]})
    b.add_node("collect", collect, defer=deferred)
    b.add_edge(START, "parts")
    b.add_edge(START, "prepare_manuals")
    b.add_edge("prepare_manuals", "manuals")
    if deferred:
        b.add_edge("parts", "collect")
        b.add_edge("manuals", "collect")
    else:
        b.add_edge(["parts", "manuals"], "collect")
    b.add_edge("collect", END)
    return b.compile()


for deferred in [False, True]:
    for part in join_graph(deferred).stream({"findings": []}, stream_mode="updates"):
        print("deferred :", deferred, "update :", part)

deferred : False update : {'prepare_manuals': None}
deferred : False update : {'parts': {'findings': [{'role': 'parts', 'data': {'part_no': 'CE-CF-02', 'name': 'traction cooling fan', 'stock': 6, 'price_eur': 420.0, 'depot': 'Rietpoort'}}]}}
deferred : False update : {'manuals': {'findings': [{'role': 'manuals', 'data': '[ce-12-error-codes.md] | F-CE-110 | traction system over-temperature | check the traction cooling fan ; replace it if airflow is below 80 percent |'}]}}
deferred : False update : {'collect': {'collected': ['manuals', 'parts']}}
deferred : True update : {'prepare_manuals': None}
deferred : True update : {'parts': {'findings': [{'role': 'parts', 'data': {'part_no': 'CE-CF-02', 'name': 'traction cooling fan', 'stock': 6, 'price_eur': 420.0, 'depot': 'Rietpoort'}}]}}
deferred : True update : {'manuals': {'findings': [{'role': 'manuals', 'data': '[ce-12-error-codes.md] | F-CE-110 | traction system over-temperature | check the traction cooling fan ; replace it if airflow is 

## Things to try

Optional : we can change the part, the context boundary or the policy and inspect the result.

In [8]:
finish_metrics("demo05")

metrics : {'model': 'openai/gpt-6-luna', 'rps': 0.06, 'calls': 1, 'input_tokens': 55, 'output_tokens': 32, 'seconds': 18.0}
